# llm_runs

> Phase 10 runs: `LLMConfig`, the world / adaptor / channel set-up for both tasks, `runLLM` (one row per episode: welfare, task metrics, interface, sharing, exploration, tokens and cost; plus the turn log), `makeClient` (fake by default; live only with `EXPLRL_LLM_LIVE=1` and a cap) and `runLLMCell` for `bench_grid.runGrid`.

In [ ]:
#| default_exp llm_runs

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
import os
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np

from explRL.bench_grid import rowKey
from explRL.bench_metrics import beliefs, discoveries, freeRiding, recovery
from explRL.board_agents import BoardLead, ClaimBoard
from explRL.harness import MessageBus
from explRL.harness_runs import benchWorld, HarnessConfig, makeChannel, makeTeam, netHarvest
from explRL.llm_agent import LLMMember, ScriptedLiarText, elicitBeliefs, runTurns
from explRL.llm_board import TextBoardEnv
from explRL.llm_budget import Budget, estimateCost
from explRL.llm_client import LLMClient, ResponseCache, loadEnvFile
from explRL.llm_env import TextForagingEnv, locationNames
from explRL.llm_fake import FakeClient, fakeReply
from explRL.multi import spawnRngs
from explRL.taskboard import TaskBoardEnv, TaskBoardSim
from explRL.rounds import runRounds
from explRL.team_time import teamColumns
from explRL.territory import homeShare, squadHomes
from explRL.team_refs import idealFree, ifdDistanceTies, ledgerArrays, occupancy, socialOptimum

NAN, DELTA = float('nan'), 0.5        # DELTA: the Phase 8 IFD tolerance


@dataclass
class LLMConfig:
    task: str = 'forage'              # 'forage' | 'board'
    channel: str = 'none'
    N: int = 4
    turns: int = 50                   # per agent (B / N in the equal-budget arm)
    B: int = 0                        # equal-budget total agent-turns (0 = off)
    model: str = 'claude-haiku-5-5'
    effort: str = 'low'
    maxTokens: int = 2000
    memory: str = 'full'
    framing: str = 'own'
    nLiar: int = 0
    episodes: int = 1                 # persistence arm
    lessons: bool = False
    difficulty: int = 2               # task board
    protocol: str = 'events'          # 'events' (runTurns) or 'rounds' (runRounds, window below)
    window: float = 0.0
    squads: int = 0                   # 0 = no squads; else k squads with home ranges (foraging)
    beta: float = 0.0                 # home bias of squad members (numpy: score bonus; inf = never leave)
    world: dict = field(default_factory=dict)   # benchWorld / TaskBoardEnv overrides; board 'solveRate' for numpy

    def asDict(self):
        return asdict(self)

    @classmethod
    def fromDict(cls, d):
        return cls(**{**d, 'world': dict(d.get('world', {}))})

    def turnsPerAgent(self):
        return self.B // self.N if self.B else self.turns


def setup(cfg, seed, numpy=False):
    "World, text adaptor and channel for one episode (`numpy`: the board's simulated-solve version)."
    N = cfg.N
    kw = {k: v for k, v in cfg.world.items() if k != 'T'}      # 'T': fixed world time of numpy cells
    if cfg.task == 'forage':
        w = benchWorld(**kw)
        w.reset(seed, N)
        return w, TextForagingEnv(w, locationNames(w.L, seed)), makeChannel(HarnessConfig(channel=cfg.channel), w.L, N, seed)
    rate = kw.pop('solveRate', 0.5)
    w = TaskBoardSim(seed, N, rate, difficulty=cfg.difficulty, **kw) if numpy else \
        TaskBoardEnv(seed, N, difficulty=cfg.difficulty, **kw)
    ch = dict(none=None, blackboard=ClaimBoard(), hierarchical=BoardLead(N),
              messages=MessageBus(N, 2, 1.0, np.inf, spawnRngs(seed + 2000, N)))[cfg.channel]
    return w, TextBoardEnv(w), ch


def taskMetrics(cfg, w, hist, honest, T, est=None):
    "Welfare per agent-turn (net of danger or penalties) and the task's own metrics."
    N, tEnd = cfg.N, max(r.t + r.dt for r in hist.records)
    if cfg.task == 'forage':
        L = ledgerArrays(w.ledger)
        rowNet, period = netHarvest(w, L), w.Tmove
        net = np.bincount(L['agent'], weights=rowNet, minlength=N)
        times, counts = occupancy(L, N, w.nPatches, tEnd)
        disc = discoveries(w, L)
        rmrt, reached = recovery(w, L, N, tEnd, DELTA)
        div, err = beliefs(w, est) if est else (NAN, NAN)
        extra = dict(rmrt=rmrt, reached=reached, discoveries=int(disc.sum()), freeRiding=freeRiding(L, disc, N),
                     ifdMean=float(ifdDistanceTies(counts, idealFree(w, N)[0], w.intake, N).mean()),
                     diversity=div, beliefError=err, soShare=float(net.sum()) / (tEnd * socialOptimum(w, N)[1]))
    else:
        net, L, rowNet, period = w.netByAgent(), ledgerArrays(w.ledger), np.array(w.netRows), 10.0
        extra = dict(solved=w.counts['solved'], opens=w.counts['opens'], wrong=w.counts['wrong'],
                     duplicates=w.duplicateTurns(), discoveries=int(w.firstSolves().sum()),
                     soShare=float(net.sum()) / (T * w.socialOptimum(1.0)))
    h = list(honest)
    return dict(welfare=float(net.sum()) / (N * T), welfareHonest=float(net[h].sum()) / (len(h) * T) if h else NAN,
                tEnd=float(tEnd), **extra, **teamColumns(L, rowNet, period, tEnd))


def llmStats(members):
    "Interface, sharing, exploration, token and cost totals over the LLM members' logs."
    recs = [x for m in members for x in m.log]
    turns = [x for x in recs if x['kind'] == 'turn']
    u = {k: sum(x['usage'].get(k, 0) for x in recs) for k in
         ('input_tokens', 'cache_creation_input_tokens', 'cache_read_input_tokens', 'output_tokens')}
    exp = [np.mean([x['explore'] for x in m.log if x['kind'] == 'turn']) for m in members if m.log]
    inp = u['input_tokens'] + u['cache_creation_input_tokens'] + u['cache_read_input_tokens']
    return dict(invalid=sum(not x['valid'] for x in turns), refusals=sum(x['refusal'] is not None for x in recs),
                maxTokenStops=sum(x['stop'] == 'max_tokens' for x in recs),
                reportShare=float(np.mean([x['report'] is not None for x in turns])) if turns else NAN,
                exploreShare=float(np.mean(exp)) if exp else NAN, exploreSpread=float(np.std(exp)) if exp else NAN,
                calls=len(recs), inputTokens=u['input_tokens'], cacheWriteTokens=u['cache_creation_input_tokens'],
                cacheReadTokens=u['cache_read_input_tokens'], outputTokens=u['output_tokens'],
                cost=float(sum(x['cost'] for x in recs)),
                promptCacheRead=u['cache_read_input_tokens'] / inp if inp else NAN)


def runLLM(cfg, seed, client):
    "One run (one row per episode) with LLM agents; the scripted liars take the first slots. -> (rows, turn log)."
    T, nLLM = cfg.turnsPerAgent(), cfg.N - cfg.nLiar
    client.reserve(estimateCost(cfg.model, cfg.task, nLLM, cfg.episodes * (T + (cfg.task == 'forage'))))
    rows, log, lessons = [], [], [''] * nLLM
    for e in range(cfg.episodes):
        ws = seed if cfg.episodes == 1 else 10000 + 100 * seed + e
        w, env, ch = setup(cfg, ws)
        liars = [ScriptedLiarText(cfg.task, ch, r) for r in spawnRngs(ws + 3000, cfg.nLiar)]
        homes = squadHomes(w.L, cfg.N, cfg.squads) if cfg.squads else [None] * cfg.N
        env.home = {i: homes[i] for i in range(cfg.N) if homes[i]}
        llm = [LLMMember(client, env, ch, cfg.memory, cfg.framing, lessons[k] if cfg.lessons else '', homes[cfg.nLiar + k],
                         strict=cfg.beta == float('inf')) for k in range(nLLM)]
        hist = runRounds(liars + llm, w, cfg.window, turns=T) if cfg.protocol == 'rounds' else runTurns(liars + llm, w, T)
        est = [elicitBeliefs(m) for m in llm] if cfg.task == 'forage' else None
        rows.append(dict(episode=e, **taskMetrics(cfg, w, hist, range(cfg.nLiar, cfg.N), T, est), **llmStats(llm),
                         **({'homeShare': homeShare(ledgerArrays(w.ledger), homes)} if cfg.squads else {})))
        log += [dict(episode=e, **x) for m in llm for x in m.log]
        lessons = [m.notes for m in llm]
    return rows, log


def makeClient(cfg, spec=None):
    "`spec`: kind 'fake' (default) or 'live' (only with EXPLRL_LLM_LIVE=1), optional cache dir, cap and spend ledger."
    spec = spec or {}
    live = spec.get('kind') == 'live'
    if live and os.environ.get('EXPLRL_LLM_LIVE') != '1':
        raise RuntimeError('live API calls need EXPLRL_LLM_LIVE=1')
    if live and not spec.get('cap'):
        raise RuntimeError('live API calls need a spending cap')
    if live:
        loadEnvFile()                          # ANTHROPIC_API_KEY from the git-ignored .env.local, if not already set
    return LLMClient(cfg.model, cfg.effort, cfg.maxTokens, ResponseCache(spec['cache']) if spec.get('cache') else None,
                     Budget(spec['cap'], spec['ledger']) if spec.get('ledger') else None,
                     client=None if live else FakeClient(fakeReply), floor=spec.get('floor', 0.0), timing=[])


def cellRow(cell, cfg, r):
    "Cell keys, the config flattened (world.* columns), then the run's metrics."
    row = dict(study=cell['study'], factor=cell['factor'], level=str(cell['level']), seed=cell['seed'],
               runner=cell.get('runner', 'llm'), budget='equal' if cfg.B else 'turns')
    for k, v in cfg.asDict().items():
        row.update({f'world.{a}': b for a, b in v.items()} if k == 'world' else {k: int(v) if isinstance(v, bool) else v})
    return {**row, **r}


def runLLMCell(cell):
    "Run one LLM cell in this process (one client per process); the turn log goes to `cell['logDir']` if given."
    cfg = LLMConfig.fromDict(cell['cfg'])
    client = makeClient(cfg, cell.get('client'))
    client.tag = rowKey(cellRow(cell, cfg, dict(episode=0)))        # timing records only
    rows, log = runLLM(cfg, cell['seed'], client)
    rows = [cellRow(cell, cfg, r) for r in rows]
    if cell.get('logDir'):
        p = Path(cell['logDir']) / 'timing' / (client.tag.replace('|', '__') + '.jsonl')
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(''.join(json.dumps(x, sort_keys=True) + '\n' for x in client.timing))
        keys = {r['episode']: rowKey(r) for r in rows}
        p = Path(cell['logDir']) / (keys[0].replace('|', '__') + '.jsonl')
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(''.join(json.dumps(dict(key=keys[x['episode']], **x), sort_keys=True) + '\n' for x in log))
    return rows

## Tests

In [ ]:
import json, math, os, tempfile
from pathlib import Path
from explRL.llm_runs import *
from explRL.llm_budget import BudgetExceeded
from explRL.llm_client import LLMClient, ResponseCache
from explRL.llm_fake import FakeClient, fakeReply

def same(a, b):
    return json.dumps(a, sort_keys=True, default=str).replace('NaN', 'nan') == json.dumps(b, sort_keys=True, default=str).replace('NaN', 'nan')

cfg = LLMConfig(task='board', channel='messages', N=8, B=200, world=dict(life=20))
assert LLMConfig.fromDict(json.loads(json.dumps(cfg.asDict()))) == cfg and cfg.turnsPerAgent() == 25
assert LLMConfig(turns=7).turnsPerAgent() == 7


def fake():
    return LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply))

# rounds and squads (speedup plan Part B): homes in the prompt, home share in the row; rounds decide on a snapshot
c = fake()
rows, log = runLLM(LLMConfig(channel='blackboard', N=4, turns=5, protocol='rounds', window=5.0, squads=2, beta=1.0,
                             world=dict(tStay=5.0)), 3, c)
assert 0 <= rows[0]['homeShare'] <= 1 and rows[0]['calls'] == 4 * 5 + 4
assert all('home range is the locations' in q['system'][0]['text'] for q in c.client.requests)
r0 = runLLM(LLMConfig(channel='blackboard', N=4, turns=5, world=dict(tStay=5.0)), 3, fake())[0][0]
assert 'homeShare' not in r0 and same(r0, runLLM(LLMConfig(channel='blackboard', N=4, turns=5, protocol='rounds', window=0.0, world=dict(tStay=5.0)), 3, fake())[0][0])

# text foraging: N agents x turns calls plus one belief call each; Phase 8 metrics where they apply
c = fake()
rows, log = runLLM(LLMConfig(channel='blackboard', N=2, turns=6), 3, c)
r = rows[0]
assert len(rows) == 1 and r['calls'] == 2 * 6 + 2 and len(c.client.requests) == 14 and len(log) == 14
assert r['invalid'] == 0 and r['refusals'] == 0 and 0 < r['reportShare'] < 1 and r['cost'] > 0 and 'cacheHit' not in r
for k in ('welfare', 'welfareHonest', 'rmrt', 'ifdMean', 'discoveries', 'freeRiding', 'diversity', 'beliefError',
          'exploreShare', 'exploreSpread', 'soShare', 'inputTokens', 'outputTokens', 'promptCacheRead', 'tEnd'):
    assert k in r, k
assert r['welfare'] == r['welfareHonest'] and sum(x['kind'] == 'beliefs' for x in log) == 2
assert abs(r['teamWelfare'] - r['welfare'] * 2 * 6) < 1e-9          # team total = per agent-turn x N x turns
assert r['tEnd'] < 40 and 'teamRate_0' not in r                       # shorter than one move period: no complete period
assert same(rows, runLLM(LLMConfig(channel='blackboard', N=2, turns=6), 3, fake())[0])      # deterministic

# the task board: solves, wrong answers, duplicate turns; no belief calls
rows, log = runLLM(LLMConfig(task='board', channel='blackboard', N=2, turns=6, difficulty=0), 3, fake())
assert rows[0]['calls'] == 12 and {'solved', 'wrong', 'duplicates', 'soShare'} <= set(rows[0]) and 'rmrt' not in rows[0]
assert abs(rows[0]['teamWelfare'] - rows[0]['welfare'] * 2 * 6) < 1e-9
r30 = runLLM(LLMConfig(task='board', channel='none', N=2, turns=30, difficulty=0), 4, fake())[0][0]
assert [k for k in r30 if k.startswith('teamRate_')] == ['teamRate_0', 'teamRate_1', 'teamRate_2']
assert abs(sum(r30[f'teamRate_{k}'] for k in range(3)) * 10 - r30['teamWelfare']) < 1e-9

# a scripted liar takes slot 0 and makes no calls; honest welfare leaves it out
for task in ('forage', 'board'):
    rows, log = runLLM(LLMConfig(task=task, channel='blackboard', N=3, nLiar=1, turns=4), 5, fake())
    assert rows[0]['calls'] == 2 * 4 + 2 * (task == 'forage') and {x['agent'] for x in log} == {1, 2}

# equal budget: B / N turns per agent
rows, _ = runLLM(LLMConfig(N=4, B=8, channel='none'), 1, fake())
assert rows[0]['calls'] == 8 + 4

# persistence: one row per episode on world seeds 10 000 + 100 s + e; lessons (last notes) carried when asked
c = fake()
rows, log = runLLM(LLMConfig(channel='blackboard', N=2, turns=3, episodes=2, lessons=True), 7, c)
assert [r['episode'] for r in rows] == [0, 1] and {x['episode'] for x in log} == {0, 1}
systems = [q['system'][0]['text'] for q in c.client.requests]
assert not any('Lessons' in s for s in systems[:8]) and all('Lessons' in s for s in systems[8:])
w, env, _ = setup(LLMConfig(), 10000 + 700 + 1)
assert env.names == locationNames(64, 10701)

# the run's estimate is checked against what is left before its first live call
tmp = Path(tempfile.mkdtemp())
c = LLMClient('claude-haiku-5-5', budget=Budget(1e-6, tmp / 'spend.jsonl'), client=FakeClient(fakeReply))
try:
    runLLM(LLMConfig(N=2, turns=50), 0, c); raise AssertionError('not refused')
except BudgetExceeded:
    assert len(c.client.requests) == 0

# with a response cache, a rerun sends nothing and gives the same rows
cache = ResponseCache(tmp / 'cache')
c1 = LLMClient('claude-haiku-5-5', cache=cache, client=FakeClient(fakeReply))
r1, l1 = runLLM(LLMConfig(channel='messages', N=3, turns=4), 2, c1)
c2 = LLMClient('claude-haiku-5-5', cache=cache, client=FakeClient(fakeReply))
r2, l2 = runLLM(LLMConfig(channel='messages', N=3, turns=4), 2, c2)
assert len(c2.client.requests) == 0 and c2.hits == c1.calls and c2.calls == 0
assert same(r1, r2) and same(l1, l2)            # rows and turn log do not depend on the cache (AC-9)

# live clients need the opt-in variable and a cap; fake is the default
os.environ.pop('EXPLRL_LLM_LIVE', None)
for spec in (dict(kind='live', cap=1.0), ):
    try:
        makeClient(LLMConfig(), spec); raise AssertionError
    except RuntimeError as e:
        assert 'EXPLRL_LLM_LIVE' in str(e)
assert isinstance(makeClient(LLMConfig()).client, FakeClient)

# a cell: keys, flattened config, turn log written per cell
cell = dict(study='main', factor='main', level='forage', N=2, seed=4, runner='llm', logDir=str(tmp / 'turns'),
            cfg=LLMConfig(channel='hierarchical', N=2, turns=3, world=dict(Tmove=30.0)).asDict())
rows = runLLMCell(cell)
assert rows[0]['study'] == 'main' and rows[0]['world.Tmove'] == 30.0 and rows[0]['budget'] == 'turns'
lines = [json.loads(x) for x in next((tmp / 'turns').glob('*.jsonl')).read_text().splitlines()]
assert len(lines) == 2 * 3 + 2 and all(x['key'] == rowKey(rows[0]) for x in lines)
assert {'obs', 'response', 'action', 'valid', 'reward', 'usage', 'cost'} <= set(lines[0]) and 'cached' not in lines[0]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()